# Mixed Galerkin (CLN bulk + HOIBC surface) -- verified results showcase

Eddy-current admittance Y(s) of a conductor by the **mixed Galerkin** formulation:
bulk Cauer Ladder Network (CLN) Foster modes (low frequency) + Higher-Order
Impedance Boundary Condition (HOIBC) Senior tower (deep skin), coupled via the
Schur complement (= discrete Steklov-Poincare / DtN operator). **Zero free
parameters** -- the bulk/surface crossover is set by the Galerkin system, not a
tuned `d`.

This notebook is a **conservative results showcase** of the two verified
smooth-boundary cases. The reusable analytic references now live in
`radia.maglev.mixed_galerkin.references`; remaining cube, square, time-domain,
and NGSolve validation scripts are promotion candidates for `docs/`, `src/`, or
`validation_test/`, not a source-code archive.

**Verified headline accuracy** (post Phase-8b reference-bug correction):

| Geometry | 1-DOF (planar SIBC) | + gamma_1 | + gamma_2 | + gamma_3 |
|---|---|---|---|---|
| Cylinder | 0.04% wall band | 2.4e-4% | 2e-5% | 1e-5% |
| Sphere   | 0.11%           | 0.001% (tower terminates) | | |
| 2D square | 0.03-0.26% (corner Mellin; slower Foster ref) | promotion pending | | |
| 3D cube  | audit pending | | | |

Refs: Senior 1962 / Mitzner 1967 / Yuferev-Ida 2010 (HOIBC tower); Kameari et al.
2018 (CLN bulk); Quarteroni-Valli 1999 (Steklov-Poincare).

## Cylinder -- Senior tower truncation

Infinite-z cylinder cross-section. Mixed Galerkin (bulk CLN + HOIBC Senior tower). VERIFIED: 1-DOF wall band 0.039%, then ~100x per added Senior DOF (2-DOF 2.4e-4%, 3-DOF 2e-5%, 4-DOF 1e-5%).

Analytic reference: `radia.maglev.mixed_galerkin.references.Y_exact_cylinder`.

In [1]:
"""
Cylinder mixed Galerkin, rank-1 bulk + N-DOF Senior tower surface.

Phase 8b of the 2026-05-28 -> 2026-06-12 research sprint, corrected.

The surface basis adds HOIBC Senior tower corrections beyond the
planar SIBC envelope of ../01_no_d_baseline.py:

    psi_1(r; s) = exp(-(a-r) t) - 1                   (planar SIBC)
    psi_2(r; s) = ((a-r) / a) exp(-(a-r) t)            (gamma_1 = -1/(2a))
    psi_3(r; s) = ((a-r) / a)^2 exp(-(a-r) t)          (gamma_2 = -1/(8 a^2))
    psi_4(r; s) = ((a-r) / a)^3 exp(-(a-r) t)          (gamma_3 = -1/(8 a^3))

This is the Taylor expansion of the cylinder Bessel I_1/I_0 asymptotic
factor sqrt(a/r) = 1 + d/(2a) + 3 d^2/(8 a^2) + ... in powers of
d = a - r.  The Galerkin coefficients automatically pick up the
canonical Senior tower factors 1/2, 1/8, etc.

## Senior tower of the cylinder (memory)

Cylinder Y/K_0 ~ 1 + sum_k b_k / (gamma a)^k with the DIVERGENT
asymptotic series

    b = [0, -1/2, -1/8, -1/8, -25/128, -13/32, -1073/1024, ...]

Senior 1962 / Mitzner 1967 / Yuferev-Ida 2010.

## Phase 8b result

With proper Y_exact (full Bessel + Senior-tower asymptotic
continuation past the scipy iv() overflow region):

   basis       | wall-band max | max anywhere
   ------------+---------------+--------------
   1-DOF       |    0.0386%    |   0.0639%
   2-DOF (g_1) |    0.00024%   |   0.0123%
   3-DOF (g_2) |    0.00002%   |   0.0027%
   4-DOF (g_3) |    0.00001%   |   0.0006%

i.e. each Senior tower DOF gives ~100x improvement at wall band.
Compare against the (Phase 7) sphere result, where Senior tower
TERMINATES at gamma_1 so 2-DOF is already optimal.
"""

from __future__ import annotations

import math
import cmath
import numpy as np
from scipy.integrate import quad

from radia.maglev.mixed_galerkin.references import (
    K_SIBC_cylinder,
    Y_DC_cylinder,
    Y_exact_cylinder,
)

SIGMA = 5.8e7
MU = 4 * math.pi * 1e-7
A = 5e-3
MS = MU * SIGMA
Y_DC = Y_DC_cylinder(A, SIGMA)
K_SIBC = K_SIBC_cylinder(A, SIGMA, MU)

K0_BB = math.pi * A**4 / 8
K1_BB = math.pi * A**6 / 48
B_B = math.pi * A**4 / 8


def _integrate_complex(f, lo, hi, t=None, limit=400):
    points = None
    if t is not None and abs(t) > 1.0:
        skin = 1.0 / abs(t)
        if skin < hi - lo:
            points = list(np.geomspace(skin / 100, min(20 * skin, (hi - lo) * 0.99), 30))
            points = [p for p in points if lo < p < hi]
    r_real, _ = quad(lambda x: f(x).real, lo, hi, limit=limit, points=points)
    r_imag, _ = quad(lambda x: f(x).imag, lo, hi, limit=limit, points=points)
    return complex(r_real, r_imag)


def psi(k: int, u, t):
    """k-th surface basis function evaluated at u = a - r.

    k = 1 ... 4 correspond to planar SIBC + Senior tower g_1 ... g_3.
    """
    if k == 1:
        return cmath.exp(-u * t) - 1
    if k == 2:
        return (u / A) * cmath.exp(-u * t)
    if k == 3:
        return (u / A)**2 * cmath.exp(-u * t)
    if k == 4:
        return (u / A)**3 * cmath.exp(-u * t)
    raise ValueError(f"unsupported k = {k}")


def dpsi(k: int, u, t):
    """Radial derivative d psi / dr at u = a - r.  Note: dr = -du."""
    if k == 1:
        return t * cmath.exp(-u * t)
    if k == 2:
        return cmath.exp(-u * t) / A * (u * t - 1)
    if k == 3:
        return (u / A**2) * cmath.exp(-u * t) * (u * t - 2)
    if k == 4:
        return (u**2 / A**3) * cmath.exp(-u * t) * (u * t - 3)
    raise ValueError(f"unsupported k = {k}")


def Y_mixed_galerkin(s, N_surf: int):
    """rank-1 bulk + N_surf-DOF surface Galerkin admittance."""
    t = cmath.sqrt(s * MS)
    dim = 1 + N_surf
    K_mat = np.zeros((dim, dim), dtype=complex)
    b_vec = np.zeros(dim, dtype=complex)

    K_mat[0, 0] = K0_BB + s * MS * K1_BB
    b_vec[0] = B_B

    for k in range(1, N_surf + 1):
        # Cross with bulk phi_0 = (a^2 - r^2)/4, grad phi_0 = -(A - u)/2 (radial).
        K0_bk = _integrate_complex(
            lambda u, kk=k: (-(A - u) / 2) * dpsi(kk, u, t) * 2 * math.pi * (A - u),
            0, A, t=t)
        K1_bk = _integrate_complex(
            lambda u, kk=k: ((2 * A * u - u**2) / 4) * psi(kk, u, t) * 2 * math.pi * (A - u),
            0, A, t=t)
        K_mat[0, k] = K0_bk + s * MS * K1_bk
        K_mat[k, 0] = K_mat[0, k]
        b_vec[k] = _integrate_complex(
            lambda u, kk=k: psi(kk, u, t) * 2 * math.pi * (A - u), 0, A, t=t)
        for j in range(1, k + 1):
            K0_kj = _integrate_complex(
                lambda u, kk=k, jj=j: dpsi(kk, u, t) * dpsi(jj, u, t) * 2 * math.pi * (A - u),
                0, A, t=t)
            K1_kj = _integrate_complex(
                lambda u, kk=k, jj=j: psi(kk, u, t) * psi(jj, u, t) * 2 * math.pi * (A - u),
                0, A, t=t)
            K_mat[k, j] = K0_kj + s * MS * K1_kj
            K_mat[j, k] = K_mat[k, j]

    xi = np.linalg.solve(K_mat, -s * MS * b_vec)
    v_avg = (xi @ b_vec) / (math.pi * A**2)
    return Y_DC * (1 + v_avg)


def main():
    print("=== Cylinder mixed Galerkin: rank-1 bulk + N-DOF Senior tower ===")
    print(f"a = {A*1e3} mm, sigma = {SIGMA:.2e} S/m, mu = {MU:.4e} H/m")
    print()
    print(f"Sample points:")
    print(f"{'f (Hz)':>10}  {'1-DOF':>10}  {'2-DOF g_1':>10}  {'3-DOF g_2':>10}  {'4-DOF g_3':>10}")
    for f in [1e3, 1e4, 5e4, 1e5, 5e5, 1e6, 1e7, 1e8]:
        s = 1j * 2 * math.pi * f
        Y_e = Y_exact_cylinder(s, A, SIGMA, MU)
        row = [
            abs(Y_e - Y_mixed_galerkin(s, N)) / abs(Y_e) * 100 for N in (1, 2, 3, 4)
        ]
        print(f"{f:10.2e}  {row[0]:8.4f}%  {row[1]:8.4f}%  {row[2]:8.4f}%  {row[3]:8.4f}%")

    print()
    print("Full sweep summary (1 Hz to 1e8 Hz, 81 points):")
    fs = np.logspace(0, 8, 81)
    wall_mask = (fs > 1e4) & (fs < 1e6)
    print(f"{'N-DOF':>6}  {'max anywhere':>16}  {'wall band max':>16}")
    for N in (1, 2, 3, 4):
        errs = []
        for f in fs:
            s = 1j * 2 * math.pi * f
            Y_e = Y_exact_cylinder(s, A, SIGMA, MU)
            errs.append(abs(Y_e - Y_mixed_galerkin(s, N)) / abs(Y_e))
        errs = np.array(errs)
        print(f"  {N}    {errs.max()*100:13.5f}%   {errs[wall_mask].max()*100:13.5f}%")

main()


=== Cylinder mixed Galerkin: rank-1 bulk + N-DOF Senior tower ===
a = 5.0 mm, sigma = 5.80e+07 S/m, mu = 1.2566e-06 H/m

Sample points:
    f (Hz)       1-DOF   2-DOF g_1   3-DOF g_2   4-DOF g_3
  1.00e+03    0.0294%    0.0107%    0.0011%    0.0001%
  1.00e+04    0.0447%    0.0005%    0.0001%    0.0000%


  5.00e+04    0.0143%    0.0000%    0.0000%    0.0000%
  1.00e+05    0.0081%    0.0000%    0.0000%    0.0000%
  5.00e+05    0.0019%    0.0000%    0.0000%    0.0000%
  1.00e+06    0.0010%    0.0000%    0.0000%    0.0000%
  1.00e+07    0.0001%    0.0000%    0.0000%    0.0000%
  1.00e+08    0.0000%    0.0000%    0.0000%    0.0000%

Full sweep summary (1 Hz to 1e8 Hz, 81 points):
 N-DOF      max anywhere     wall band max


  1          0.06386%         0.03855%


  2          0.01233%         0.00024%


  3          0.00270%         0.00002%


  4          0.00063%         0.00001%


## Sphere -- HOIBC gamma_1 curvature correction

Solid sphere. The gamma_1 = -1/a curvature correction gives ~100x improvement. VERIFIED: 1-DOF wall band 0.114% -> 2-DOF 0.0011% (102x).

Analytic reference: `radia.maglev.mixed_galerkin.references.Y_exact_sphere`.

In [2]:
"""
Sphere mixed Galerkin with HOIBC gamma_1 = -1/a curvature correction.

Phase 7 of the 2026-05-28 -> 2026-06-12 research sprint.

Adds a SECOND surface basis function reflecting the leading
curvature correction in the Senior tower:

    psi_1(r; s) = exp(-(a-r) t) - 1        (planar SIBC, same as 01)
    psi_2(r; s) = ((a-r) / a) exp(-(a-r) t) (curvature, gamma_1 = -1/a)

The (a-r)/a factor is the leading Taylor expansion term of the
(a/r) geometric factor in the sphere Bessel asymptote
    v(r) ~ -1 + (a/r) exp(-gamma (a-r)).

## Sphere Senior tower terminates at gamma_1

For the sphere, the Senior tower coefficients are b = [0, -1, 0, 0, ...]
(constructive derivation in memory
 project_senior_hoibc_tower_constructive_derivation.md).  Adding gamma_1
captures the entire non-trivial Senior tower; higher-order DOFs would
add nothing.  This is why the 2-DOF sphere result hits machine
precision at deep skin (1e-8 % at f = 10^8 Hz).

## Result

    1-DOF                            2-DOF (gamma_1)
    --------                         ----------------
    wall band max =  0.114%          wall band max =  0.0011% (100x improvement)
    max anywhere  =  0.137%          max anywhere  =  0.0037%
"""

from __future__ import annotations

import math
import cmath
import numpy as np
from scipy.integrate import quad

from radia.maglev.mixed_galerkin.references import (
    K_SIBC_sphere,
    Y_DC_sphere,
    Y_exact_sphere,
)

SIGMA = 5.8e7
MU = 4 * math.pi * 1e-7
A = 5e-3
MS = MU * SIGMA
V_SPH = (4.0 / 3.0) * math.pi * A**3
Y_DC = Y_DC_sphere(A, SIGMA)
K_SIBC = K_SIBC_sphere(A, SIGMA, MU)

K0_BB = 4 * math.pi * A**5 / 45
K1_BB = 8 * math.pi * A**7 / 945
B_B = 4 * math.pi * A**5 / 45


def _integrate_complex(f, lo, hi, t=None, limit=400):
    points = None
    if t is not None and abs(t) > 1.0:
        skin = 1.0 / abs(t)
        if skin < hi - lo:
            points = list(np.geomspace(skin / 100, min(20 * skin, (hi - lo) * 0.99), 30))
            points = [p for p in points if lo < p < hi]
    r_real, _ = quad(lambda x: f(x).real, lo, hi, limit=limit, points=points)
    r_imag, _ = quad(lambda x: f(x).imag, lo, hi, limit=limit, points=points)
    return complex(r_real, r_imag)


def Y_mixed_galerkin(s, with_gamma1: bool):
    """rank-1 bulk + 1-or-2-DOF surface.  with_gamma1=False reproduces 01."""
    t = cmath.sqrt(s * MS)

    K0_ss1 = _integrate_complex(
        lambda u: t**2 * cmath.exp(-2 * u * t) * 4 * math.pi * (A - u)**2, 0, A, t=t)
    K1_ss1 = _integrate_complex(
        lambda u: (cmath.exp(-u * t) - 1)**2 * 4 * math.pi * (A - u)**2, 0, A, t=t)
    K0_bs1 = _integrate_complex(
        lambda u: (-(A - u) / 3) * (t * cmath.exp(-u * t)) * 4 * math.pi * (A - u)**2, 0, A, t=t)
    K1_bs1 = _integrate_complex(
        lambda u: ((2 * A * u - u**2) / 6) * (cmath.exp(-u * t) - 1) * 4 * math.pi * (A - u)**2,
        0, A, t=t)
    b_s1 = _integrate_complex(
        lambda u: (cmath.exp(-u * t) - 1) * 4 * math.pi * (A - u)**2, 0, A, t=t)

    if not with_gamma1:
        K_mat = np.array(
            [
                [K0_BB + s * MS * K1_BB, K0_bs1 + s * MS * K1_bs1],
                [K0_bs1 + s * MS * K1_bs1, K0_ss1 + s * MS * K1_ss1],
            ],
            dtype=complex,
        )
        b_vec = np.array([B_B, b_s1], dtype=complex)
        xi = np.linalg.solve(K_mat, -s * MS * b_vec)
        v_avg = (xi[0] * B_B + xi[1] * b_s1) / V_SPH
        return Y_DC * (1 + v_avg)

    # 2-DOF surface: add psi_2 = (u/a) exp(-u t).
    # psi_2'_r = exp(-ut)/a * (u t - 1)
    K0_ss2 = _integrate_complex(
        lambda u: (cmath.exp(-u * t) / A * (u * t - 1))**2 * 4 * math.pi * (A - u)**2, 0, A, t=t)
    K1_ss2 = _integrate_complex(
        lambda u: ((u / A) * cmath.exp(-u * t))**2 * 4 * math.pi * (A - u)**2, 0, A, t=t)
    K0_ss12 = _integrate_complex(
        lambda u: t * cmath.exp(-u * t) * (cmath.exp(-u * t) / A * (u * t - 1)) *
        4 * math.pi * (A - u)**2, 0, A, t=t)
    K1_ss12 = _integrate_complex(
        lambda u: (cmath.exp(-u * t) - 1) * ((u / A) * cmath.exp(-u * t)) *
        4 * math.pi * (A - u)**2, 0, A, t=t)
    K0_bs2 = _integrate_complex(
        lambda u: (-(A - u) / 3) * ((1 / A) * cmath.exp(-u * t) * (u * t - 1)) *
        4 * math.pi * (A - u)**2, 0, A, t=t)
    K1_bs2 = _integrate_complex(
        lambda u: ((2 * A * u - u**2) / 6) * ((u / A) * cmath.exp(-u * t)) *
        4 * math.pi * (A - u)**2, 0, A, t=t)
    b_s2 = _integrate_complex(
        lambda u: ((u / A) * cmath.exp(-u * t)) * 4 * math.pi * (A - u)**2, 0, A, t=t)

    K11 = K0_ss1 + s * MS * K1_ss1
    K22 = K0_ss2 + s * MS * K1_ss2
    K12 = K0_ss12 + s * MS * K1_ss12
    K_b = K0_BB + s * MS * K1_BB
    K_b1 = K0_bs1 + s * MS * K1_bs1
    K_b2 = K0_bs2 + s * MS * K1_bs2

    K_mat = np.array(
        [[K_b, K_b1, K_b2], [K_b1, K11, K12], [K_b2, K12, K22]], dtype=complex
    )
    b_vec = np.array([B_B, b_s1, b_s2], dtype=complex)
    xi = np.linalg.solve(K_mat, -s * MS * b_vec)
    v_avg = (xi @ b_vec) / V_SPH
    return Y_DC * (1 + v_avg)


def main():
    print("=== Sphere mixed Galerkin: 1-DOF vs 2-DOF (HOIBC gamma_1 = -1/a) ===")
    print(f"a = {A*1e3} mm, sigma = {SIGMA:.2e} S/m, mu = {MU:.4e} H/m")
    print(f"gamma_1 = -1/a = {-1/A:.4e}")
    print()

    print(f"{'f (Hz)':>10}  {'|Y_exact|':>12}  {'err_1':>10}  {'err_2 (gamma_1)':>16}")
    for f in [1.0, 1e3, 1e4, 5e4, 1e5, 1e6, 1e7, 1e8]:
        s = 1j * 2 * math.pi * f
        Y_e = Y_exact_sphere(s, A, SIGMA, MU)
        e1 = abs(Y_e - Y_mixed_galerkin(s, False)) / abs(Y_e)
        e2 = abs(Y_e - Y_mixed_galerkin(s, True)) / abs(Y_e)
        print(f"{f:10.2e}  {abs(Y_e):12.4e}  {e1*100:8.4f}%  {e2*100:13.6f}%")

    print()
    fs = np.logspace(0, 8, 81)
    e1_all, e2_all = [], []
    for f in fs:
        s = 1j * 2 * math.pi * f
        Y_e = Y_exact_sphere(s, A, SIGMA, MU)
        e1_all.append(abs(Y_e - Y_mixed_galerkin(s, False)) / abs(Y_e))
        e2_all.append(abs(Y_e - Y_mixed_galerkin(s, True)) / abs(Y_e))
    e1_all = np.array(e1_all)
    e2_all = np.array(e2_all)
    wall = (fs > 1e4) & (fs < 1e6)
    print(f"Full sweep:")
    print(f"  1-DOF  max anywhere = {e1_all.max()*100:.4f}%,  wall band = {e1_all[wall].max()*100:.4f}%")
    print(f"  2-DOF  max anywhere = {e2_all.max()*100:.4f}%,  wall band = {e2_all[wall].max()*100:.4f}%")
    print(f"  Wall-band improvement factor: {e1_all[wall].max()/e2_all[wall].max():.1f}x")

main()


=== Sphere mixed Galerkin: 1-DOF vs 2-DOF (HOIBC gamma_1 = -1/a) ===
a = 5.0 mm, sigma = 5.80e+07 S/m, mu = 1.2566e-06 H/m
gamma_1 = -1/a = -2.0000e+02

    f (Hz)     |Y_exact|       err_1   err_2 (gamma_1)
  1.00e+00    3.0369e+01    0.0000%       0.000000%
  1.00e+03    2.2176e+01    0.0056%       0.002774%
  1.00e+04    7.9719e+00    0.1251%       0.001735%
  5.00e+04    3.6971e+00    0.0500%       0.000095%
  1.00e+05    2.6369e+00    0.0294%       0.000027%
  1.00e+06    8.4586e-01    0.0039%       0.000000%
  1.00e+07    2.6870e-01    0.0004%       0.000000%
  1.00e+08    8.5091e-02    0.0000%       0.000000%



Full sweep:
  1-DOF  max anywhere = 0.1371%,  wall band = 0.1142%
  2-DOF  max anywhere = 0.0037%,  wall band = 0.0011%
  Wall-band improvement factor: 102.2x


## Remaining promotion candidates

These are still in `examples/mixed_galerkin/` and should be promoted in later batches:

- `square2d/01_corner_envelope.py` -- 2D square corner-aware tensor envelope
  psi = f(x) f(y) (0.03-0.26%); slower because of the large Foster reference sum.
- `cube3d/01..07` -- 3D cube rank-N + closed K_ss + NGSolve FEM ground truth
  (Foster reference audit still open -> provisional).
- `time_domain/01_cube_aaa_step_response.py` -- AAA rational fit -> stable poles
  (no `d` tuning), the time-domain Cauer realization direction.
- `ngsolve_validation/` -- framework-agnostic NGSolve FEM cross-validation.
- `_references/` -- remaining square/cube analytic references pending src/API promotion.
